# Significance Tests — Proposed Ensemble vs Each Single-Model Baseline
Loads the cached predictions from Part 1 and Part 2, and runs McNemar's test
comparing the proposed ensemble (Swin + ViT + ResNet50) against each of the
6 single models individually -- matching the standard "proposed vs baseline"
comparison table style used in published papers.

No GPU needed for this notebook -- it's pure analysis on saved arrays.

## Cell 1 — Setup

In [ ]:
!pip install statsmodels -q
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import json as jsonlib
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Load all cached predictions

In [ ]:
cache = '/content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache'

pred_swin = np.load(f'{cache}/pred_swin.npy')
pred_vit = np.load(f'{cache}/pred_vit.npy')
pred_resnet = np.load(f'{cache}/pred_resnet.npy')
pred_densenet = np.load(f'{cache}/pred_densenet.npy')
pred_efficientnet = np.load(f'{cache}/pred_efficientnet.npy')
pred_levit = np.load(f'{cache}/pred_levit.npy')
ensemble_probs = np.load(f'{cache}/ensemble_probs.npy')
true_labels = np.load(f'{cache}/true_labels.npy')
with open(f'{cache}/class_names.json') as f:
    class_names = jsonlib.load(f)

ensemble_preds = np.argmax(ensemble_probs, axis=1)
print("All predictions loaded. Test set size:", len(true_labels))

All predictions loaded. Test set size: 2538


## Cell 3 — Run McNemar's test: proposed ensemble vs each single model

In [ ]:
baselines = {
    "Swin": pred_swin,
    "ViT": pred_vit,
    "LeViT": pred_levit,
    "ResNet50": pred_resnet,
    "DenseNet121": pred_densenet,
    "EfficientNetB0": pred_efficientnet,
}

ensemble_correct = (ensemble_preds == true_labels)
results = []

for name, preds in baselines.items():
    baseline_labels = np.argmax(preds, axis=1)
    baseline_correct = (baseline_labels == true_labels)

    n10 = int(np.sum(baseline_correct & ~ensemble_correct))   # baseline correct, ensemble wrong
    n01 = int(np.sum(~baseline_correct & ensemble_correct))   # baseline wrong, ensemble correct
    n11 = int(np.sum(baseline_correct & ensemble_correct))
    n00 = int(np.sum(~baseline_correct & ~ensemble_correct))

    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=(n10 + n01 < 25), correction=True)

    baseline_acc = np.mean(baseline_correct)
    results.append({
        "Comparison": f"Proposed vs. {name}",
        "Baseline Acc.": f"{baseline_acc*100:.2f}%",
        "Statistic": round(float(result.statistic), 4),
        "p-value": round(float(result.pvalue), 6) if result.pvalue >= 0.0001 else "<0.0001",
        "Sig. (a=0.05)": "Yes" if result.pvalue < 0.05 else "No",
    })

results_df = pd.DataFrame(results)
print(f"Proposed ensemble accuracy: {np.mean(ensemble_correct)*100:.2f}%\n")
print(results_df.to_string(index=False))

Proposed ensemble accuracy: 98.86%

                 Comparison Baseline Acc.  Statistic   p-value Sig. (a=0.05)
          Proposed vs. Swin        98.42%     6.0000   0.03469           Yes
           Proposed vs. ViT        98.23%     8.0357  0.004586           Yes
         Proposed vs. LeViT        95.15%    80.0833   <0.0001           Yes
      Proposed vs. ResNet50        97.95%    13.8286    0.0002           Yes
   Proposed vs. DenseNet121        96.89%    38.7258   <0.0001           Yes
Proposed vs. EfficientNetB0        97.68%    21.0250   <0.0001           Yes


## Cell 4 — Save results table to Drive

In [ ]:
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs'
results_df.to_csv(f'{output_folder}/significance_test_results.csv', index=False)
print(f"Saved to: {output_folder}/significance_test_results.csv")

Saved to: /content/drive/MyDrive/thesis_ensemble_outputs/significance_test_results.csv
